# TradeAI PRO — Autonomous ML Model Trainer & Database Publisher

This notebook trains the **Tier-3 AI Ensemble Stack** (XGBoost + LightGBM + Calibrator) on historical NSE Equity or Crypto data, validates accuracy metrics, and packages the model bundle directly to your Supabase PostgreSQL database (`model_bundle:v1`).

### Compatibility:
- **Kaggle Notebooks**: CPU or T4/P100 GPU (Free 30 hrs/week, supports background "Save & Run All")
- **Google Colab**: Free CPU or T4 GPU runtime

### Security:
- **Zero Hardcoded Secrets**: `DATABASE_URL` is fetched from Kaggle Secrets or Colab Secrets. Never paste passwords in code cells.

## 1. Setup Environment & Clone Repository

In [ ]:
import os, sys

# Clone repository if not running directly inside the project root
if not os.path.exists("server.py"):
    !git clone https://github.com/Himanshugupta2024/TradeAI-PRO.git trading
    %cd trading
else:
    print("Already in TradeAI-PRO workspace.")

print("Current directory:", os.getcwd())
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

# Install dependencies
!pip install --upgrade pip
!pip install -r requirements.txt

## 2. Load Supabase Secrets (Kaggle or Colab)

In [ ]:
# Securely fetch DATABASE_URL without hardcoding credentials in the notebook
db_url = os.environ.get("DATABASE_URL")

if not db_url:
    try:
        # Kaggle Secrets
        from kaggle_secrets import UserSecretsClient
        user_secrets = UserSecretsClient()
        db_url = user_secrets.get_secret("DATABASE_URL")
        print("Loaded DATABASE_URL from Kaggle Secrets.")
    except Exception:
        pass

if not db_url:
    try:
        # Google Colab Secrets
        from google.colab import userdata
        db_url = userdata.get("DATABASE_URL")
        print("Loaded DATABASE_URL from Google Colab Secrets.")
    except Exception:
        pass

if db_url:
    os.environ["DATABASE_URL"] = db_url
    print("DATABASE_URL successfully configured for Supabase!")
else:
    print("WARNING: DATABASE_URL not found. Models will be trained locally, but cannot be published to Supabase.")
    print("Add 'DATABASE_URL' to Kaggle (Add-ons -> Secrets) or Colab (Secrets tab).")

## 3. Configure Training Parameters

In [ ]:
# Asset Selection:
# 'nse' for Indian Equities (RELIANCE, TCS, HDFCBANK, INFY, ICICIBANK, SBIN)
# 'crypto' for Digital Assets (BTC/USDT, ETH/USDT, SOL/USDT)
ASSET_TYPE = "nse"

# Timeframe and Lookback
TIMEFRAME = "1h"
LOOKBACK_DAYS = 365

# Quick Mode (skip heavy Optuna search for fast 5-minute training)
QUICK_MODE = True

# Custom symbol override (leave empty for default Tier-1 leaders)
SYMBOLS = "RELIANCE,TCS,HDFCBANK,INFY,ICICIBANK,SBIN"

print(f"Training Target: {ASSET_TYPE.upper()} | Timeframe: {TIMEFRAME} | Days: {LOOKBACK_DAYS} | Quick: {QUICK_MODE}")

## 4. Run Training Pipeline

In [ ]:
import subprocess
import json

cmd = [
    sys.executable,
    "scripts/train_and_publish.py",
    "--asset-type", ASSET_TYPE,
    "--symbols", SYMBOLS,
    "--timeframe", TIMEFRAME,
    "--days", str(LOOKBACK_DAYS),
    "--source", "kaggle_notebook" if "kaggle" in sys.modules else "colab_notebook",
]

if QUICK_MODE:
    cmd.append("--quick")
else:
    cmd.extend(["--optuna", "--trials", "15"])

# If database URL is present, publish bundle
if os.environ.get("DATABASE_URL"):
    cmd.append("--publish")

print("Executing command:", " ".join(cmd))
process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in process.stdout:
    print(line, end="")

process.wait()
if process.returncode == 0:
    print("\n Training completed successfully!")
else:
    print(f"\n Training failed with exit code {process.returncode}")

## 5. Verify Model Artifacts & Gate Compliance

In [ ]:
from pathlib import Path
from api.model_sync import current_status, read_bundle_from_db

print("=== Local Disk Artifacts ===")
local_status = current_status()
print("Models Directory:", local_status["models_dir"])
print("Files Present:", local_status["files_present"])

print("\n=== Supabase Remote Bundle Status ===")
try:
    bundle = read_bundle_from_db()
    if bundle:
        print("Published Bundle Version:", bundle.get("version"))
        print("Trained At:", bundle.get("trained_at"))
        print("Source:", bundle.get("source"))
        print("Total Bytes (compressed):", bundle.get("total_bytes"))
        print("Validation Accuracy Metric:", bundle.get("metrics", {}).get("val_accuracy"))
        print("Files in Bundle:", list(bundle.get("files", {}).keys()))
    else:
        print("No bundle found in Supabase runtime_state.")
except Exception as e:
    print("Could not query Supabase bundle:", e)